# Experiment 2 — a bad size estimate, with and without AQE

`orders_big` (24 M rows) joined with the *gold* customers of `customers_big`: a table that is large on disk, filtered down to a few thousand rows.
Spark UI: http://localhost:4040 → **SQL / DataFrame** (open each query; with AQE there is an initial and a final plan).

In [1]:
import os, time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m4-exp2-bad-estimate").getOrCreate()
fact = spark.read.parquet("/opt/data/m3/orders_big")
cb = spark.read.parquet("/opt/data/m4/customers_big")

size_mb = sum(os.path.getsize(p.replace("file:", "")) for p in cb.inputFiles()) / 2**20
print(f"customers_big on disk: {size_mb:.0f} MiB in {len(cb.inputFiles())} files")

gold = cb.filter(F.col("tier") == "gold")
print("gold rows:", gold.count())

def timed(label, df):
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 13:57:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


customers_big on disk: 147 MiB in 8 files


[Stage 2:=============================>                             (4 + 4) / 8]

gold rows: 5000


## Run 1 — AQE off

**PREDICT:** the join strategy of the static plan, and why.

In [2]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
j1 = fact.join(gold, "customer_id")
j1.explain("formatted")
timed("Run 1 AQE off", j1)

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]

(3) Filter [codegen id : 1]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(4) Exchange
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3

[Stage 7:==========================================>            (156 + 8) / 200]

Run 1 AQE off: 5.3s


What the planner believed about the size of each side:

In [3]:
j1.explain("cost")

== Optimized Logical Plan ==
Project [customer_id#1, order_id#0L, shop_id#2, amount#3, order_date#4, note#5, tier#7, country#8, profile_a#9, profile_b#10], Statistics(sizeInBytes=143.5 PiB)
+- Join Inner, (customer_id#1 = customer_id#6), Statistics(sizeInBytes=147.7 PiB)
   :- Filter isnotnull(customer_id#1), Statistics(sizeInBytes=1028.9 MiB)
   :  +- Relation [order_id#0L,customer_id#1,shop_id#2,amount#3,order_date#4,note#5] parquet, Statistics(sizeInBytes=1028.9 MiB)
   +- Filter ((isnotnull(tier#7) AND (tier#7 = gold)) AND isnotnull(customer_id#6)), Statistics(sizeInBytes=147.0 MiB)
      +- Relation [customer_id#6,tier#7,country#8,profile_a#9,profile_b#10] parquet, Statistics(sizeInBytes=147.0 MiB)

== Physical Plan ==
*(5) Project [customer_id#1, order_id#0L, shop_id#2, amount#3, order_date#4, note#5, tier#7, country#8, profile_a#9, profile_b#10]
+- *(5) SortMergeJoin [customer_id#1], [customer_id#6], Inner
   :- *(2) Sort [customer_id#1 ASC NULLS FIRST], false, 0
   :  +- Exchan

## Run 2 — AQE on

**PREDICT:** what `explain()` shows before running; what the final plan in the SQL tab shows after; which work AQE can still avoid at that point; runtime relative to Run 1.

In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
j2 = fact.join(gold, "customer_id")
j2.explain("formatted")
timed("Run 2 AQE on", j2)

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan parquet  (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan parquet  (5)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(3) Exchange
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: hashpartitioning(customer_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=245]

(4) Sort
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amou

Run 2 AQE on: 2.6s


## Run 3 — AQE on, a filter that keeps only three customers

**PREDICT:** the final plan's join strategy.

In [6]:
three = cb.filter(F.col("customer_id").isin(1, 2, 3))
j3 = fact.join(three, "customer_id")
timed("Run 3 three keys", j3)

[Stage 20:============================>                            (8 + 8) / 16]

Run 3 three keys: 0.8s


## Stop

In [ ]:
spark.stop()